# Q3 — S&P 500 Median Correction Drawdown (≥5%) & Durations
**Homework:** `homework1.md:63-94` — *Median drawdown of significant (≥5%) S&P 500 corrections.*

**Novice:** Find ATH = `Close == cummax`, then min between ATHs, drawdown formula, percentiles.

**Answer:** **Median 7.99%** (p25 6.23 / p75 14.02), peak→trough median 40.5d, 74 significant corrections. Top 10 matches hint exactly.


## 3.1 Download long history

`^GSPC` from 1950 → ~19k days. Check shape.

In [ ]:
import yfinance as yf, pandas as pd
close = yf.download("^GSPC", start="1950-01-01", progress=False, auto_adjust=False)
if isinstance(close.columns, pd.MultiIndex): close = close["Close"]["^GSPC"]
else: close = close["Close"]
close = close.dropna()
print(len(close), close.index.min().date(), "→", close.index.max().date())
close.tail(3)

## 3.2 Find ATH points

`cummax()` is running max; `Close == cummax` marks new highs. Count them.

In [ ]:
cummax = close.cummax()
is_ath = close == cummax
ath_dates = close.index[is_ath]
ath_vals = close[is_ath]
print(f"ATH count: {len(ath_dates)}")
print(ath_dates[:3].tolist())
print(ath_vals.head(3).to_string())

## 3.3 For each ATH pair, find low between

Loop consecutive ATHs → window `(high, next_high)` → min, dates, drawdown, durations.

In [ ]:
records=[]
for i in range(len(ath_dates)-1):
    high_date, high = ath_dates[i], float(ath_vals.iloc[i])
    next_high = ath_dates[i+1]
    window = close[(close.index>high_date) & (close.index<next_high)]
    if window.empty: continue
    low, low_date = float(window.min()), window.idxmin()
    dd = (high-low)/high*100
    records.append({"high_date":high_date, "low_date":low_date, "high":high, "low":low, "drawdown":dd, "dur_pt":(low_date-high_date).days, "dur_pp":(next_high-high_date).days, "next_high":next_high})
df = pd.DataFrame(records)
print(f"Corrections: {len(df)}")
df.head(3)

## 3.4 Filter ≥5% and percentiles

- `df[df.drawdown>=5]` → significant
- `quantile([0.25,0.5,0.75])` for drawdown and durations.

In [ ]:
sig = df[df["drawdown"]>=5]
print(f"Significant ≥5%: {len(sig)}")
print(sig.sort_values("drawdown", ascending=False).head(10).to_string(index=False))
for col in ["drawdown","dur_pt","dur_pp"]:
    print(col, sig[col].quantile([0.25,0.5,0.75]).to_string())
print(f"\nAnswer Q3 median drawdown {sig['drawdown'].median():.2f}%")

## 3.5 Validate against hint top 10 (homework1.md:84-94)

Our `dur_pt` (peak→trough) matches hint exactly.

In [ ]:
hints=[("2007-10-09",56.8,517),("2000-03-24",49.1,929),("2020-02-19",33.9,33)]
for h,exp_dd,exp_dur in hints:
    r=df[df["high_date"]==pd.Timestamp(h)].iloc[0]
    print(h, f"calc {r['drawdown']:.2f}% vs {exp_dd}%", f"dur {r['dur_pt']} vs {exp_dur}")